# Module 1: Customer Dimension Data Cleaning & Demographic Standardization
**Project:** Multi-Source Retail Analytics & Anomaly Detection Pipeline  
**Author:** Piyush Kumar Singh  
**Stack:** Python 3.12+, Pandas, NumPy  
**Input:** `customers_messy.csv` (~6,060 raw customer records)  
**Output:** `customers_clean.csv` (Sanitized Customer Dimension)

---

### Executive Summary & Business Objective
The customer dimension table (`customers_messy.csv`) defines customer profiles, geographical location, registration dates, and loyalty tiers for over 6,000 accounts. In raw transactional systems, customer extracts suffer from several hygiene defects:
1. **Unstructured Datetimes:** Registration timestamps stored as unstructured object strings.
2. **Text Inconsistencies & Leading/Trailing Whitespace:** Mixed case city names (e.g., `'delhi'`, `'Delhi '`) and string literals representing nulls (`'Nan'`).
3. **Demographic Outliers:** Impossible age records (e.g. `age > 100`) caused by entry errors or mock placeholder dates.
4. **Missing Attributes:** Incomplete email addresses and geographical fields.

This module builds an automated, robust data cleaning pipeline to sanitize data types, handle missing values with statistically sound imputations, and output a production-grade dimension table (`customers_clean.csv`).

## 1. Environment Setup & Library Initialization
Import core analytical packages: `pandas` for data wrangling and `numpy` for numerical array operations and NaN representations.

In [2]:
import numpy as np 
import pandas as pd

## 2. Ingesting Raw Customer Extract & Initial Schema Profiling
Load `customers_messy.csv` and inspect raw dimensional structure, initial rows, null value distribution, and column data types.

In [3]:
df=pd.read_csv('customers_messy.csv')

In [4]:
df.head()

,customer_id,customer_name,email,city,country,signup_date,age,customer_segment
0,CUST10000,Priya Sharma,user0@example.com,Ranchi,India,2023-12-23,70.0,Regular
1,CUST10001,Amit Gupta,user1@example.com,Chennai,India,2024-12-24,47.0,New
2,CUST10002,Ananya Kumar,user2@example.com,Pune,India,2019-12-25,53.0,Regular
3,CUST10003,Priya Singh,user3@example.com,Hyderabad,India,2020-09-07,38.0,Premium
4,CUST10004,Isha Reddy,user4@example.com,Hyderabad,India,2019-04-04,68.0,Regular


In [5]:
df.shape

(6060, 8)

In [6]:
df.isnull().sum()

customer_id           0
customer_name         0
email               150
city                181
country               0
signup_date           0
age                 237
customer_segment      0
dtype: int64

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6060 entries, 0 to 6059
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       6060 non-null   str    
 1   customer_name     6060 non-null   str    
 2   email             5910 non-null   str    
 3   city              5879 non-null   str    
 4   country           6060 non-null   str    
 5   signup_date       6060 non-null   str    
 6   age               5823 non-null   float64
 7   customer_segment  6060 non-null   str    
dtypes: float64(1), str(7)
memory usage: 378.9 KB


In [8]:
df.dtypes

customer_id             str
customer_name           str
email                   str
city                    str
country                 str
signup_date             str
age                 float64
customer_segment        str
dtype: object

## 3. Temporal Feature Normalization
Convert `signup_date` from an unstructured object string into standardized `datetime64[ns]` format. This enables downstream cohort analysis, customer tenure calculations, and lifecycle tracking.

In [9]:
df['signup_date']=pd.to_datetime(df['signup_date'])

In [10]:
df.dtypes

customer_id                    str
customer_name                  str
email                          str
city                           str
country                        str
signup_date         datetime64[us]
age                        float64
customer_segment               str
dtype: object

## 4. Categorical Cleaning: City Dimension Hygiene
Customer city records contain mixed casing, extraneous whitespace, and literal string `'Nan'` values.
We apply a 4-step standardization pipeline:
1. Strip leading and trailing whitespace (`.str.strip()`)
2. Standardize casing to Title Case (`.str.title()`)
3. Replace string literals `'Nan'` with true floating-point `np.nan`
4. Apply **Mode Imputation** (`df['city'].mode()[0]`) to fill missing geographical entries with the most representative market location.

In [11]:
df['city'].unique()

<StringArray>
[    'Ranchi',    'Chennai',       'Pune',  'Hyderabad',     'Jaipur',
      'Delhi',    'Kolkata',    'Lucknow',  'Bengaluru',     'Mumbai',
          nan,   'CHENNAI ', 'HYDERABAD ',    'MUMBAI ',    'JAIPUR ',
   'LUCKNOW ',    'RANCHI ',      'PUNE ', 'BENGALURU ',     'DELHI ',
   'KOLKATA ',       'NAN ']
Length: 22, dtype: str

In [12]:
df['city']=df['city'].str.strip().str.title()

In [13]:
df['city'].unique()

<StringArray>
[   'Ranchi',   'Chennai',      'Pune', 'Hyderabad',    'Jaipur',     'Delhi',
   'Kolkata',   'Lucknow', 'Bengaluru',    'Mumbai',         nan,       'Nan']
Length: 12, dtype: str

In [14]:
df['city']=df['city'].replace('Nan',np.nan)

In [15]:
df['city'].unique()

<StringArray>
[   'Ranchi',   'Chennai',      'Pune', 'Hyderabad',    'Jaipur',     'Delhi',
   'Kolkata',   'Lucknow', 'Bengaluru',    'Mumbai',         nan]
Length: 11, dtype: str

In [16]:
df['city']=df['city'].fillna(df['city'].mode()[0])

## 5. Country Identifier Integrity Audit
Audit the `country` column to confirm referential integrity and ensure no unrecognized multi-country anomalies exist in this domestic e-commerce dataset.

In [17]:
df['country'].unique()

<StringArray>
['India', 'USA', 'UK', 'UAE']
Length: 4, dtype: str

## 6. Demographic Outlier Remediation & Age Imputation
Inspect the `age` distribution. Human ages greater than 100 in consumer e-commerce represent data entry corruption:
- Identify and replace all values where `age > 100` with `np.nan`.
- Impute missing age observations using the **median** age rather than the mean to prevent distortion from demographic skew.

In [18]:
df['age'].unique()

array([ 70.,  47.,  53.,  38.,  68.,  22.,  33.,  74.,  45.,  27.,  41.,
        32.,  31.,  40.,  69.,  61.,  44.,  56.,  72.,  21.,  60.,  nan,
        46.,  43.,  36.,  20.,  51.,  71.,  30.,  37.,  52.,  63.,  19.,
        55.,  48., 130.,  50.,  73.,  29.,  42.,  25.,  26.,  64.,  67.,
        34.,  58.,  59.,  57.,  49.,  54.,  24.,  39.,  23.,  65.,  28.,
        18.,  66., 110.,  35.,  62.,   8.])

In [19]:
df.loc[(df['age']>100),'age']=np.nan

In [20]:
df['age'].isnull().sum()

np.int64(294)

In [21]:
df['age']=df['age'].fillna(df['age'].median())

## 7. Customer Loyalty Tiers & Contact Information Verification
Inspect customer loyalty tiers (`customer_segment`) and replace missing email strings with `'Not Available'` to ensure auditability without row loss.

In [22]:
df['customer_segment'].unique()

<StringArray>
['Regular', 'New', 'Premium', 'VIP']
Length: 4, dtype: str

In [23]:
df.columns

Index(['customer_id', 'customer_name', 'email', 'city', 'country',
       'signup_date', 'age', 'customer_segment'],
      dtype='str')

In [29]:
df['email']=df['email'].fillna('Not Available')

## 8. Final Null Audit & Production Clean Export
Execute a final missing value scan across all customer dimension attributes to confirm 100% data completeness, then persist the sanitized dataset to `customers_clean.csv`.

In [30]:
df.isnull().sum()

customer_id         0
customer_name       0
email               0
city                0
country             0
signup_date         0
age                 0
customer_segment    0
dtype: int64

In [31]:
df.to_csv('customers_clean.csv',index=False)